In [ ]:
import pandas as pd 
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
conn=sqlite3.connect("/Users/shyamchauhan/Desktop/home/codes/project/vendor_invoice_intelligence_portal/data/inventory.db")
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print("Table name : ",table)
    df=pd.read_sql_query(f"select * from {table} limit 5",conn)
    # display(df)

In [ ]:
vendor_df=pd.read_sql_query("select * from vendor_invoice",conn)
vendor_df.head()

In [ ]:
vendor_df[['Quantity','Freight','Dollars']].corr() # corr is used to find correlation coefficient
vendor_df.head()

In [ ]:
# relation between Quantity , Freight ,Dollars

plt.figure(figsize=(4,2))
sns.heatmap(vendor_df[['Quantity','Freight','Dollars']].corr(),annot=True)
plt.show()

plt.scatter(vendor_df['Quantity'],vendor_df['Freight'])
plt.scatter(vendor_df['Dollars'],vendor_df['Freight'])
plt.legend(['Quantity','Dollars'])
plt.ylabel('Freight cost')
plt.show()

In [ ]:
vendor_df['freight_cost_per_unit']=vendor_df['Freight']/vendor_df['Quantity']
vendor_df

In [ ]:
low_quantity=vendor_df['Quantity'].quantile(0.25)
high_quantity=vendor_df['Quantity'].quantile(0.75)

In [ ]:
vendor_df.loc[vendor_df['Quantity']<low_quantity,'freight_cost_per_unit'].mean()

In [ ]:
vendor_df.loc[vendor_df['Quantity']>high_quantity,'freight_cost_per_unit'].mean() 
# mean is less implies that vendor pays less when orders high quantity

In [ ]:
X=vendor_df[['Dollars']]
Y=vendor_df['Freight']

In [ ]:
vendor_df.describe().round()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [ ]:
model1=LinearRegression()
model1.fit(X_train,Y_train)

model2=DecisionTreeRegressor(max_depth=4 ,random_state=42)
model2.fit(X_train,Y_train)

model3=RandomForestRegressor(max_depth=4,random_state=42)
model3.fit(X_train,Y_train)

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error

def evaluate_model(model,X_test,Y_test,model_name):
    preds=model.predict(X_test)

    mae=mean_absolute_error(Y_test,preds)
    rmse = np.sqrt(mean_squared_error(Y_test, preds))
    # rmse=root_mean_squared_error(Y_test,preds,squared=False)
    r2=r2_score(Y_test,preds)*100

    print(f"\n{model_name} Performance : ")
    print(f"MAE : {mae:.2f}")
    print(f"RMSE : {rmse : .2f}")
    print(f"R_square : {r2:.2f}%")


In [ ]:
evaluate_model(model1,X_test,Y_test,'Linear Regression')
evaluate_model(model2,X_test,Y_test,'Decision Tree Regression')
evaluate_model(model3,X_test,Y_test,'Random Forest Regression')

In [ ]:
plt.scatter(X_test,Y_test)
plt.plot(X_test,model1.predict(X_test),color='red')

In [ ]:
input_data={
    "Dollars":[18500,9000]
}
df=pd.DataFrame(input_data)

In [ ]:
model1.predict(df)